 get_feature_names_out(input_features)
  np.array([cols to return],dtype= "object")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = "/content/drive/MyDrive/2026 AI class/dataset/penguins_size.csv"

In [ ]:
import pandas as pd
df = pd.read_csv(file_path)
df.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

trf1 = ColumnTransformer(
    [
    ('impute_numerical',SimpleImputer(strategy="mean"),["culmen_length_mm","culmen_depth_mm","flipper_length_mm","body_mass_g"]
     )
    ],
    remainder="passthrough",verbose_feature_names_out=False).set_output(transform = "pandas")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class GenderImputer(BaseEstimator,TransformerMixin):
  def fit(self,x):
    self.clean = x[x["sex"].isin(["MALE","FEMALE"])]
    self.mass_median = self.clean.groupby(["species","sex"])["body_mass_g"].median()
    return self

  def transform(self,x):

    def fill_empty(row):
      spec = row["species"]
      male_med = self.mass_median[spec]["MALE"]
      female_med = self.mass_median[spec]["FEMALE"]

      threshold = (male_med +female_med)/2

      if row['body_mass_g'] >= threshold:
        return "MALE"
      else:
        return "FEMALE"

    self.mask = x["sex"].isna()

    x.loc[self.mask,"sex"] = x[self.mask].apply(fill_empty, axis = 1)
    return x[["species","sex","body_mass_g"]]


  def get_feature_names_out(self, input_features = None):
    return np.array(["species","sex","body_mass_g"], dtype = "object")




In [ ]:
trf2 = ColumnTransformer([("impute_catagorical",GenderImputer(),["species","body_mass_g","sex"])],
        remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")

In [ ]:
from sklearn.base import BaseEstimator,TransformerMixin

class CulmenRatio(BaseEstimator, TransformerMixin):
  def fit(self,x,y = None):
    return self
  def transform(self,x):
    x["culmen_ratio"] = x["culmen_length_mm"] / x["culmen_depth_mm"]
    return x


In [ ]:
from sklearn.preprocessing import OneHotEncoder
trf3 = ColumnTransformer([("ohe",OneHotEncoder(sparse_output = False),["species","island","sex"])],
                         remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")

In [ ]:
from sklearn.preprocessing import MinMaxScaler
trf4 = ColumnTransformer([
    ("scaling",MinMaxScaler(),["culmen_length_mm","culmen_depth_mm","flipper_length_mm","body_mass_g","culmen_ratio"])
],remainder="passthrough",verbose_feature_names_out=False).set_output(transform=  "pandas")

In [ ]:
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ("trans1",trf1),
    ("trans2",trf2),
    ("culmenratio",CulmenRatio()),
    ("trans3",trf3),
    ("trans4",trf4)
])

In [ ]:
pipe

Pipeline(steps=[('trans1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_numerical',
                                                  SimpleImputer(),
                                                  ['culmen_length_mm',
                                                   'culmen_depth_mm',
                                                   'flipper_length_mm',
                                                   'body_mass_g'])],
                                   verbose_feature_names_out=False)),
                ('trans2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_catagorical',
                                                  GenderImputer(),
                                                  ['species', 'body_ma...
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['species', 'island',
                                                   'sex'])],
                                   verbose_feature_names_out=False)),
                ('trans4',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scaling', MinMaxScaler(),
                                                  ['culmen_length_mm',
                                                   'culmen_depth_mm',
                                                   'flipper_length_mm',
                                                   'body_mass_g',
                                                   'culmen_ratio'])],
                                   verbose_feature_names_out=False))])

In [ ]:
import numpy as np
df["sex"] = df["sex"].replace(".", np.nan)

In [ ]:
df_transformed = pipe.fit_transform(df)

In [ ]:
df_transformed

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,culmen_ratio,species_Adelie,species_Chinstrap,species_Gentoo,island_Biscoe,island_Dream,island_Torgersen,sex_FEMALE,sex_MALE
0,0.254545,0.666667,0.152542,0.291667,0.228651,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,0.269091,0.511905,0.237288,0.305556,0.319487,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,0.298182,0.583333,0.389831,0.152778,0.303659,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,0.429888,0.482282,0.490088,0.417154,0.466864,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.167273,0.738095,0.355932,0.208333,0.132672,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,0.429888,0.482282,0.490088,0.417154,0.466864,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
340,0.534545,0.142857,0.728814,0.597222,0.827688,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
341,0.665455,0.309524,0.847458,0.847222,0.795990,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
342,0.476364,0.202381,0.677966,0.694444,0.716847,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
